# Семинар 6 - Память

Сегодня в программе:
1. Мотивация для создания виртуальной памяти
2. Устройство виртуальной памяти
3. `mmap` и его друзья
4. Кэши

У физической памяти есть несколько проблем:
* Она не всегда доступна одним непрерывным отрезком 
* Нет изоляции 
* Нет защиты от фрагментации 
* Сложное устройство: например, есть кэши, swap

Эти минусы показывают необходмость в создании абстракции над физической памятью. Эта абстракция называется виртуальная память. 

Процессы работают именно с виртуальной памятью и воспринимают ее как абстракцию. При обращении к памяти процесса, процессор транслирует виртуальный адрес в физический.

<img src="media/translation.png" alt="virtual memory translation" width="700" style="background-color:white;"/>

Виртуальная память на архитектуре x86 имеет следующие особенности:
* страницы 4Кб (есть hugepages размером 2Мб и 1Гб)
* 4-уровневая адресация, позволяющая адресовать 2^48 байт (256 Тб)
* таблицы адресов свои у каждого процесса. Указатель на начало первой таблицы хранится в регистре CR3
* есть кэш трансляции: TLB (Translation Lookaside Buffer)
* трансляцией занимается MMU (Memory Management Unit)

<img src="media/mapping.png" alt="virtual memory mapping" width="700" style="background-color:white;"/>

Для работы с виртуальной памятью есть системный вызов `mmap`:

```c
#include <sys/mman.h>
void* mmap(void *addr, size_t len, int prot, int flags, int fd, off_t offset);
```

Выделить `size` байт памяти с помощью него можно так:

```c
void* p = mmap(0, size, PROT_READ|PROT_WRITE, MAP_PRIVATE|MAP_ANONYMOUS, -1, 0);
```

Также с помощью `mmap` можно отображать файл в память. Это значит, что будет выделен отрезок виртуальной памяти, ассоциированный с памятью на диске. Память с диска будет загружена не сразу, а по мере необходимости. В этом случае при обращении к памяти, которая не загружена в оперативную память, будет сгенерировано "исключение", которое будет перехвачено ядром. В этом случае ядро загрузит страницу из файла в оперативную память и повторит обращение.

In [13]:
!gcc ./snippets/mmap/mmap_demo.c -o ./snippets/mmap/mmap_demo.out
!cat ./snippets/mmap/mmap_demo.c

#include <stdio.h>
#include <memory.h>
#include <unistd.h>
#include <fcntl.h>
#include <sys/stat.h>
#include <sys/mman.h>
#include <sys/time.h>
#include <sys/resource.h>
#include <sys/mman.h>

int main(int argc, char** argv) {
    int input_file = open(argv[1], O_RDWR);
    struct stat file_stats;
    fstat(input_file, &file_stats);
    char* content_ptr = mmap(NULL, file_stats.st_size, PROT_READ | PROT_WRITE, MAP_SHARED, input_file, 0);

    for (size_t i = 0; i < file_stats.st_size; i++) {
        content_ptr[i] = '\0';
    }

    strcpy(content_ptr + 15, "Hello, world!");

    munmap(content_ptr, file_stats.st_size);
    close(input_file);
}


In [14]:
!cd snippets/mmap && python3 gen_big_file.py

In [15]:
!./snippets/mmap/mmap_demo.out

У `mmap` есть парный сискол - `unmap`.

Когда вы маппите файл с `MAP_SHARED`, нет гарантии, что после `unmap` данные будут сразу сброшены на диск. Для сброса на диск есть сисколы `msync` и `fsync`. Однако и они не дают 100% гарантии: они обещают лишь то, что данные будут отправлены физическому устройству, но нет гарантии, что оно обязательно их запишет (может, например, сохранить в своем кэше). 

В частности, вот выдержка из `man` Apple по `fsync`:

```
     Note that while fsync() will flush all data from the host to the drive
     (i.e. the "permanent storage device"), the drive itself may not physi-cally physically
     cally write the data to the platters for quite some time and it may be
     written in an out-of-order sequence.

     Specifically, if the drive loses power or the OS crashes, the application
     may find that only some or none of their data was written.  The disk
     drive may also re-order the data so that later writes may be present,
     while earlier writes are not.

     This is not a theoretical edge case.  This scenario is easily reproduced
     with real world workloads and drive power failures.

     For applications that require tighter guarantees about the integrity of
     their data, Mac OS X provides the F_FULLFSYNC fcntl.  The F_FULLFSYNC
     fcntl asks the drive to flush all buffered data to permanent storage.
     Applications, such as databases, that require a strict ordering of writes
     should use F_FULLFSYNC to ensure that their data is written in the order
     they expect.  Please see fcntl(2) for more detail.
```

Еще по этой теме:
* [1](https://eclecticlight.co/2022/02/18/how-can-you-trust-a-disk-to-write-data/)
* [2](https://sqlite.org/forum/info/b94afa45dda82aae8cbf49f9d511a00b332870fc926cba18954acd889bbfb7cd)
* [Apple](https://news.ycombinator.com/item?id=30370551)
* [Linux](https://puzpuzpuz.dev/the-secret-life-of-fsync)

Есть еще пара сисколов для аллокации памяти - `brk`/`sbrk`:

```c
#include <unistd.h>

int brk(void *addr);
void *sbrk(intptr_t increment);
```

In [22]:
!gcc snippets/brk/brk.c -O0 -o snippets/brk/brk.out
!cat snippets/brk/brk.c

snippets/brk/brk.c:8:9: warning: 'sbrk' is deprecated [-Wdeprecated-declarations]
    8 |     b = sbrk(0);
      |         ^
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/unistd.h:586:1: note: 'sbrk' has been explicitly marked deprecated here
  586 | __deprecated __WATCHOS_PROHIBITED __TVOS_PROHIBITED
      | ^
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/sys/cdefs.h:214:40: note: expanded from macro '__deprecated'
  214 | #define __deprecated    __attribute__((__deprecated__))
      |                                        ^
snippets/brk/brk.c:15:5: warning: 'brk' is deprecated [-Wdeprecated-declarations]
   15 |     brk(b);
      |     ^
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/unistd.h:542:1: note: 'brk' has been explicitly marked deprecated here
  542 | __deprecated __WATCHOS_PROHIBITED __TVOS_PROHIBITED
      | ^
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/sys/cdefs.h:214:40: note: expanded from macro 

In [25]:
#!./snippets/brk/brk.out

`malloc` и `calloc` внутри себя аллоцируют память именно через `mmap` и `brk`

Чтобы посмотреть на замаппленные регионы памяти, можно почитать файл `/proc/{PID}/maps`

Поговорим подробнее про исключения, возникающие при отсутствии страницы в оперативной памяти. Они бывают 2 видов:
* **Minor page fault** - случается, когда требуемая страница *есть* в физической оперативной памяти, но не замапплена в виртуальную память процесса. Такое может произойти, например, если другой процесс загрузил требуемую страницу в память
* **Major page fault** - случается, когда требуемая страницы отсутствует в физической оперативной памяти, и её необходимо загрузить туда с диска. Занимает много времени. Такое может произойти, когда страницы впервые запрашивается процессом и отсутствует в кэше. Другой пример: если требуемая страница была вытеснена из оперативной памяти в swap файл

Теперь обсудим еще один важный аспект памяти - кэши. Кэш - это очень быстрая память, расположенная в/близко к процессору. Обычно есть 3 уровня кэша, увеличивающихся в размере и уменьшающихся в скорости. Посмотреть кэши на своей системе можно командой:
```bash
getconf -a | grep CACHE
```
или
```bash
lscpu | grep cache
```
Вывод будет примерно такой:
```bash
L1d cache: 192 KiB (6 instances)
L1i cache: 192 KiB (6 instances)
L2 cache:  3 MiB (6 instances)
L3 cache:  32 MiB (1 instance)
```

Если требуемая память находится в кэше, доступ к ней произойдет заметно быстрее, чем если бы она была в ОЗУ. При работе с памятью, участки, с которыми мы оперируем, автоматически добавляются в кэш по размеру кэшлинии (обычно 64 байта). 

* [Сравнение латентности памяти по годам](https://colin-scott.github.io/personal_website/research/interactive_latency.html)
* [Latency numbers every programmer should know](https://gist.github.com/jboner/2841832)

К примеру, посмотрим на зависимость латентности доступа к памяти в зависимости от размера аллокации:

In [26]:
!cat snippets/latency/list_traversal.cpp

//  Created by Emil Ernerfeldt on 2014-04-17.

#include <iostream>
#include <algorithm>
#include <vector>
#include <chrono>
#include <cmath>
#include <numeric> // iota
#include <random>

using namespace std;
using namespace chrono;

using Clock = chrono::steady_clock;

using Int = uint64_t;

std::random_device rd;
std::mt19937_64 g(rd());

static void clobber() {
	asm volatile("" : : : "memory");
}

struct Node {
	Int payload; // ignored; just for plausability.
	Node* next = nullptr;
};

static_assert(sizeof(Node) == 16, "Not 64-bit? That's OK too.");


// Returns nanoseconds per element.
double time(Int N, Int iters) {
	// Allocate all the memory continuously so we aren't affected by the particulars of the system allocator:
	vector<Node> memory(N);

	// Initialize the node pointers:
	vector<Node*> nodes(N);
	for (Int i=0; i<N; ++i) {
		nodes[i] = &memory[i];
	}
	//std::iota(begin(nodes), end(nodes), memory.data());

	// Randomize so emulate a list that has been shuffled around a bit.


<img src="media/memory_latency_benchmark.png" alt="memory latency caches" width="700" style="background-color:white;"/>

Латентность при случайном паттерне доступа растет, как только данные перестают помещаться в кэш. При этом при последовательном досутпе латентность наоборот снижается. Это происходит из-за того, что при обращении к 1 байту памяти в кэш загружается не он 1, а вся кэшлиния (64 байта). Это можно проверить на следующем примере:

In [27]:
!cat snippets/latency/cache_line_benchmark.cpp

#include <string>
#include <chrono>
#include <iostream>
#include <vector>
#include <random>
#include <algorithm>
//#include <immintrin.h>
#include "alligned_allocator.hpp"

// Prevent optimization
static void escape(void *p) {
    asm volatile("" : : "g"(p) : "memory");
}

static void clobber() {
    asm volatile("" : : : "memory");
}

// Cache line size (typically 64 bytes on modern systems)
constexpr size_t CACHE_LINE_SIZE = 64;
constexpr size_t INTS_PER_CACHE_LINE = CACHE_LINE_SIZE / sizeof(int);

// 1. Varying stride within cache line
void benchmark_stride(int* memory, size_t size, int stride, int iterations) {
    for (int iter = 0; iter < iterations; iter++) {
        for (size_t i = 0; i < size; i += stride) {
            memory[i]++;
        }
        clobber();
    }
    escape(memory);
}

// 2. Access every element in cache line before moving to next
void benchmark_cache_line_fully(int* memory, size_t size, int iterations) {
    size_t elements = size / sizeof(int);
    for (

<img src="media/cache_line_benchmark.png" alt="cache line benchmark with different strides" width="700" style="background-color:white;"/>

Видно, что пока данные помещаются в кэш, использование большого шага дает заметное преимущество в скорости (хотя и не такое большое, как можно было бы ожидать). Но когда данные уже не помещаются в кэш, мажорирующим фактором становится не 16 инкрементов, а доступ к памяти

На работу с памятью можно смотреть и с такой (шуточной?) стороны, что сложность обращения к памяти при аллокации `$n$` байт не `O(1)`, а `O(\sqrt{n})`. Посмотрим на более широкий график:

<img src="media/mem_latency.png" alt="wide mem latency" width="700" style="background-color:white;"/>

Здесь обе оси прологарифмированы, синий график - латентность обращения к памяти, а оранжевый - график функции корня. Вертикальными линиями отмечены границы L1, L2, L3 кэшей и оперативной памяти соответственно. У такого взгляда даже есть определенные [теоретические основания](http://www.ilikebigbits.com/2014_04_28_myth_of_ram_2.html).

Пример, как знание о кэшах можно использовать на практике. Напишем функцию для умножения матриц:

In [28]:
!cat snippets/matmul/naive.cpp

#include <iostream>
#include <chrono>
#include <vector>

template <int N>
using matrix = std::array<std::array<int, N>, N>;

template <int N>
void naive(matrix<N>& m1, matrix<N>& m2, matrix<N>& res) {
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            for (int k = 0; k < N; ++k) {
                res[i][j] = res[i][j] + m1[i][k] * m2[k][j];
            }
        }
    }
}

template <int N>
void fast(matrix<N>& m1, matrix<N>& m2, matrix<N>& res) {
    matrix<N> m2T;
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            m2T[i][j] = m2[j][i];
        }
    }
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            for (int k = 0; k < N; ++k) {
                res[i][j] = res[i][j] + m1[i][k] * m2T[j][k];
            }
        }
    }
}

int main() {
    constexpr int N = 700;
    matrix<N> m1, m2, res;

    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            m1[i][j] = ran

Получаем, что от перестановки циклов начинаем более эффективно использовать кэши, и получаем ускорение на ~30%, несмотря на то, что добавили цикл на `O(n^2)` действий. Также такой паттерн доступа к памяти позволяет компилятору использовать векторные инструкции

Еще почитать по теме:
* [Memory latency](https://en.algorithmica.org/hpc/cpu-cache/latency/)
* [Matrix multiplication](https://en.algorithmica.org/hpc/algorithms/matmul/)